# BoM Explosion — FIN material production hierarchy

For each FIN material, walks `produced_material -> component_material` down to the leaves
(`ADD`/`RM`) and flattens the whole chain into one row per `(material, component)` pair,
tagged with the root FIN's attributes. Two independent implementations: pandas and SQL
(DuckDB `WITH RECURSIVE`, queried directly on the pandas DataFrame, no DB server needed).
Result is aggregated to annual and exported to `.xlsx`.


In [ ]:
import pandas as pd
import duckdb

INPUT_FILE = "task_2_data_ex.xlsx"
OUTPUT_FILE = "bom_explosion_result.xlsx"

OUTPUT_COLUMNS = [
    "plant",
    "fin_material_id", "fin_material_release_type", "fin_material_production_type",
    "fin_production_quantity",
    "prod_material_id", "prod_material_release_type", "prod_material_production_type",
    "prod_material_production_quantity",
    "component_id", "component_material_release_type", "component_material_production_type",
    "component_consumption_quantity",
    "year",
]


## 1. Load data

In [ ]:
def load_raw_data(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)

    # Light defensive cleanup: coerce quantities/ids, then fail loudly if that introduced NaNs
    # (source file is already clean, this just guards against a bad future extract).
    qty_cols = ["produced_material_quantity", "component_material_quantity"]
    for c in qty_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    id_cols = ["plant_id", "produced_material", "component_material"]
    for c in id_cols:
        df[c] = df[c].astype(str).str.strip()
    assert not df[qty_cols + id_cols].isna().any().any(), "Unexpected NaN after cleanup"

    return df


raw = load_raw_data(INPUT_FILE)
print("shape:", raw.shape)
raw.head(3)


## 2. Pandas explosion

Explode per `(plant, year, month)` first (BoM structure is monthly), then sum to annual.

In [ ]:
def build_component_index(df: pd.DataFrame) -> dict:
    index: dict = {}
    for row in df.itertuples(index=False):
        index.setdefault((row.plant_id, row.year, row.month, row.produced_material), []).append(row)
    return index


def explode_single_fin(fin_anchor, index: dict) -> list:
    """BFS walk of one FIN material's hierarchy for one (plant, year, month)."""
    plant, year, month = fin_anchor.plant_id, fin_anchor.year, fin_anchor.month
    out, queue, visited = [], [fin_anchor.level1_material_id], set()

    while queue:
        material_id = queue.pop(0)
        if material_id in visited:
            continue  # cycle guard; also avoids re-expanding a shared sub-assembly twice
        visited.add(material_id)

        for comp in index.get((plant, year, month, material_id), []):
            out.append({
                "plant": plant,
                "fin_material_id": fin_anchor.fin_material_id,
                "fin_material_release_type": fin_anchor.fin_material_release_type,
                "fin_material_production_type": fin_anchor.fin_material_production_type,
                "fin_production_quantity": fin_anchor.fin_production_quantity,
                "prod_material_id": comp.produced_material,
                "prod_material_release_type": comp.produced_material_release_type,
                "prod_material_production_type": comp.produced_material_production_type,
                "prod_material_production_quantity": comp.produced_material_quantity,
                "component_id": comp.component_material,
                "component_material_release_type": comp.component_material_release_type,
                "component_material_production_type": comp.component_material_production_type,
                "component_consumption_quantity": comp.component_material_quantity,
                "year": year,
            })
            if comp.component_material_release_type == "PROD":  # only PROD can have sub-components
                queue.append(comp.component_material)

    return out


def explode_bom_pandas(df: pd.DataFrame) -> pd.DataFrame:
    fin = df.loc[df.produced_material_release_type == "FIN"].rename(columns={
        "produced_material": "fin_material_id",
        "produced_material_release_type": "fin_material_release_type",
        "produced_material_production_type": "fin_material_production_type",
        "produced_material_quantity": "fin_production_quantity",
        "component_material": "level1_material_id",
    })
    index = build_component_index(df)

    rows = []
    for anchor in fin.itertuples(index=False):
        rows.extend(explode_single_fin(anchor, index))
    monthly = pd.DataFrame(rows, columns=OUTPUT_COLUMNS + ["month"])

    # Annual aggregation: sum monthly quantities per (plant, fin, prod material, component, year).
    group_cols = [c for c in OUTPUT_COLUMNS if not c.endswith("_quantity")]
    annual = (
        monthly.groupby(group_cols, as_index=False, dropna=False)
        .agg(
            fin_production_quantity=("fin_production_quantity", "sum"),
            prod_material_production_quantity=("prod_material_production_quantity", "sum"),
            component_consumption_quantity=("component_consumption_quantity", "sum"),
        )[OUTPUT_COLUMNS]
    )
    return annual.sort_values(
        ["plant", "year", "fin_material_id", "prod_material_id", "component_id"]
    ).reset_index(drop=True)


result_pandas = explode_bom_pandas(raw)
print("shape:", result_pandas.shape)
result_pandas.head(10)


## 3. SQL (DuckDB `WITH RECURSIVE`)

Same logic, standard SQL, runnable as-is as a PostgreSQL view; final `GROUP BY` does the same annual sum.

In [ ]:
BOM_EXPLOSION_SQL = """
WITH RECURSIVE explosion AS (
    -- NB: month is carried through the recursion (needed so joins don't match
    -- materials across different months) and only dropped in the final SUM/GROUP BY.
    SELECT
        f.plant_id AS plant,
        f.produced_material AS fin_material_id,
        f.produced_material_release_type AS fin_material_release_type,
        f.produced_material_production_type AS fin_material_production_type,
        f.produced_material_quantity AS fin_production_quantity,
        m.produced_material AS prod_material_id,
        m.produced_material_release_type AS prod_material_release_type,
        m.produced_material_production_type AS prod_material_production_type,
        m.produced_material_quantity AS prod_material_production_quantity,
        m.component_material AS component_id,
        m.component_material_release_type AS component_material_release_type,
        m.component_material_production_type AS component_material_production_type,
        m.component_material_quantity AS component_consumption_quantity,
        f.year, f.month
    FROM raw f
    JOIN raw m
      ON m.plant_id = f.plant_id AND m.year = f.year AND m.month = f.month
     AND m.produced_material = f.component_material
    WHERE f.produced_material_release_type = 'FIN'

    UNION ALL

    SELECT
        e.plant, e.fin_material_id, e.fin_material_release_type, e.fin_material_production_type,
        e.fin_production_quantity,
        nxt.produced_material, nxt.produced_material_release_type, nxt.produced_material_production_type,
        nxt.produced_material_quantity,
        nxt.component_material, nxt.component_material_release_type, nxt.component_material_production_type,
        nxt.component_material_quantity,
        e.year, e.month
    FROM explosion e
    JOIN raw nxt
      ON nxt.plant_id = e.plant AND nxt.year = e.year AND nxt.month = e.month
     AND nxt.produced_material = e.component_id
    WHERE e.component_material_release_type = 'PROD'  -- only PROD components expand further
)
SELECT
    plant, fin_material_id, fin_material_release_type, fin_material_production_type,
    SUM(fin_production_quantity) AS fin_production_quantity,
    prod_material_id, prod_material_release_type, prod_material_production_type,
    SUM(prod_material_production_quantity) AS prod_material_production_quantity,
    component_id, component_material_release_type, component_material_production_type,
    SUM(component_consumption_quantity) AS component_consumption_quantity,
    year
FROM explosion
GROUP BY plant, fin_material_id, fin_material_release_type, fin_material_production_type,
         prod_material_id, prod_material_release_type, prod_material_production_type,
         component_id, component_material_release_type, component_material_production_type, year
ORDER BY plant, year, fin_material_id, prod_material_id, component_id;
"""

result_sql = duckdb.sql(BOM_EXPLOSION_SQL).df()
print("shape:", result_sql.shape)
result_sql.head(10)


## 4. Validation

Completeness check only (not a correctness proof): confirms nothing was lost or duplicated at the FIN level, plus basic shape checks.

In [ ]:
def validate(raw_df: pd.DataFrame, result_df: pd.DataFrame) -> None:
    # Checks that the set of (plant, year, fin_material_id) in the result exactly matches
    # the set of FIN materials in the source (nothing missing, nothing extra).
    source_fins = raw_df.loc[raw_df.produced_material_release_type == "FIN",
                              ["plant_id", "year", "produced_material"]].drop_duplicates()
    source_fins.columns = ["plant", "year", "fin_material_id"]
    result_fins = result_df[["plant", "year", "fin_material_id"]].drop_duplicates()
    assert set(map(tuple, source_fins.values)) == set(map(tuple, result_fins.values)), \
        "FIN material set mismatch between source and result"

    assert list(result_df.columns) == OUTPUT_COLUMNS, "Unexpected output column order"
    assert result_df[["plant", "year", "fin_material_id", "prod_material_id", "component_id"]].notna().all().all(), \
        "Null values in key columns"

    print("Validation passed.")


validate(raw, result_pandas)


## 5. Export

In [ ]:
result_pandas.to_excel(OUTPUT_FILE, index=False)
print(f"Saved {len(result_pandas)} rows to {OUTPUT_FILE}")
